# Part 4: Deep Reinforcement Learning — DQN, Policy Gradient, PPO & TRPO

**Course**: Reinforcement Learning for AI/ML Engineers  
**Duration**: 2 Hours  
**Instructor**: Mehdi Lotfinejad  

---

## Learning Objectives

By the end of this section, you will be able to:

1. **Explain** why tabular Q-Learning fails on large/continuous state spaces and how neural networks solve this
2. **Implement DQN** (Deep Q-Network) from scratch with Experience Replay and a Target Network
3. **Build REINFORCE** — the foundational policy gradient algorithm — and understand why it works
4. **Understand Actor-Critic** architecture and the Advantage function
5. **Implement PPO** (Proximal Policy Optimization) — the algorithm behind ChatGPT's RLHF training
6. **Understand TRPO** (Trust Region Policy Optimization) and why PPO replaced it in practice
7. **Compare** all deep RL algorithms and choose the right one for your use case
8. **Train agents** on CartPole and LunarLander using modern deep RL techniques

---


## 1. From Q-Tables to Neural Networks — Why Deep RL?

In Part 3, we implemented Q-Learning using a **table**: one row per state, one column per action.

This works perfectly for small environments like:
- FrozenLake: **16 states** × 4 actions = 64 values
- Taxi: **500 states** × 6 actions = 3,000 values
- CliffWalking: **48 states** × 4 actions = 192 values

But the real world isn't that simple:

| Environment | State Space | Q-Table Size |
|---|---|---|
| FrozenLake 4×4 | 16 discrete | 64 values — trivial |
| Atari Pong | 210×160×3 pixels | $\approx 10^{70}$ states — impossible |
| CartPole | 4 continuous values | Infinite states — impossible |
| Humanoid robot control | 376 continuous dims | Completely intractable |

### The Solution: Approximate Q with a Neural Network

Instead of storing Q(s, a) in a table, we train a neural network $Q_\theta(s, a)$ to **approximate** the Q-function:

```
Q-Table:          State → Look up row → Q-values for all actions
                  Works for small discrete state spaces only

Deep Q-Network:   State → Neural Network → Q-values for all actions
                  Works for ANY state representation (pixels, numbers, etc.)
```

**Same Q-Learning algorithm. Same update rule. Just replace the table with a network.**

### The Taxonomy of Modern RL

```
                    DEEP REINFORCEMENT LEARNING
                              |
          ┌───────────────────┼───────────────────┐
          |                   |                   |
   Value-Based         Policy-Based          Actor-Critic
   (learn Q or V)    (learn π directly)    (learn both)
          |                   |                   |
         DQN              REINFORCE            A2C / A3C
       Double DQN        TRPO / PPO              SAC
       Dueling DQN                              TD3 / DDPG
          |
      (Part 3 was        (Section 4)          (Section 5–6)
       table-based)
```

This part covers the shaded algorithms. Let's start with DQN.


In [1]:
# ============================================================
# Setup — install and import all dependencies for Part 4
# ============================================================
# Run this cell first!

import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

# PyTorch (CPU)
try:
    import torch
    print(f"PyTorch already installed: {torch.__version__}")
except ImportError:
    print("Installing PyTorch (CPU)...")
    install('torch')
    import torch
    print(f"PyTorch installed: {torch.__version__}")

# Box2D (needed for LunarLander in Section 8)
try:
    import Box2D
    print(f"Box2D already installed")
except ImportError:
    print("Installing Box2D dependencies (swig + gymnasium[box2d])...")
    install('swig')
    install('gymnasium[box2d]')
    print("Box2D installed")

# Core imports for this notebook
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical

import numpy as np
import random
import gymnasium as gym
from collections import deque, namedtuple

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print(f"\n{'='*50}")
print(f"  PyTorch  : {torch.__version__}")
print(f"  Numpy    : {np.__version__}")
print(f"  Gymnasium: {gym.__version__}")
print(f"  Device   : {'GPU (CUDA)' if torch.cuda.is_available() else 'CPU'}")
print(f"{'='*50}")
print("\nAll dependencies ready. Let's build Deep RL from scratch!")


PyTorch already installed: 2.10.0

  PyTorch  : 2.10.0
  Numpy    : 2.4.2
  Gymnasium: 1.2.3
  Device   : CPU

All dependencies ready. Let's build Deep RL from scratch!


## 2. Deep Q-Network (DQN)

DQN was introduced by DeepMind in 2013 and published in *Nature* in 2015. It was the first algorithm to master Atari games from raw pixels — a milestone moment for AI.

**Core idea**: Train a neural network $Q_\theta(s, a)$ using the same Q-Learning update:

$$\mathcal{L}(\theta) = \mathbb{E}\left[\left(\underbrace{r + \gamma \max_{a'} Q_{\theta^-}(s', a')}_{\text{TD Target (frozen)}} - Q_\theta(s, a)\right)^2\right]$$

But naïve gradient descent on this is **unstable**. DeepMind added two critical innovations:

### Innovation 1: Experience Replay 🔁

Without replay, the agent trains on consecutive transitions $(s_t, a_t, r_t, s_{t+1})$. These are **highly correlated** — like trying to learn English from one book read front-to-back without shuffling.

```
Problem:  s0→s1→s2→s3→... (sequential, correlated)
           ↓   ↓   ↓
           train on consecutive samples → unstable gradients

Solution: Store all transitions in a "replay buffer"
          Sample RANDOM mini-batches → breaks correlation → stable learning

Replay Buffer:  [(s12, a, r, s13), (s47, a, r, s48), (s3, a, r, s4), ...]
                                         ↑
                                  Random shuffle!
```

### Innovation 2: Target Network 🎯

In Q-Learning, the target $r + \gamma \max_{a'} Q(s', a')$ changes every update step — you're chasing a moving target!

```
Problem:  Target = r + γ·max Q_θ(s', a')
          θ changes → target changes → training oscillates

Solution: Keep a FROZEN copy of the network (θ⁻) for computing targets.
          Update θ⁻ ← θ  only every C steps (e.g., every 100 updates).

Online network  θ  : updated every step    ← learns
Target network  θ⁻ : frozen for C steps    ← provides stable targets
```

### DQN Architecture

```
State (continuous) → [FC Layer] → [ReLU] → [FC Layer] → [ReLU] → Q(s, a₁), Q(s, a₂), ..., Q(s, aₙ)

CartPole:  4 inputs → 64 → ReLU → 64 → ReLU → 2 outputs (Left, Right)
```


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import random
import gymnasium as gym
from collections import deque

# ============================================================
# DQN on CartPole-v1
# ============================================================

# CartPole: balance a pole on a cart
#   State : [cart_pos, cart_vel, pole_angle, pole_angular_vel]  (4 continuous)
#   Action: 0 = push left | 1 = push right
#   Reward: +1 for every step the pole stays up
#   Done  : pole falls > 12° OR cart moves > 2.4 units OR step >= 500

# ── 1. Neural Network ──────────────────────────────────────
class QNetwork(nn.Module):
    """Simple 2-hidden-layer Q-network."""
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, action_dim)
        )
    def forward(self, x):
        return self.net(x)

# ── 2. Experience Replay Buffer ────────────────────────────
class ReplayBuffer:
    """Fixed-size circular buffer storing (s, a, r, s', done) transitions."""
    def __init__(self, capacity=20_000):
        self.buf = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buf.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        s, a, r, ns, d = zip(*batch)
        return (torch.FloatTensor(np.array(s)),
                torch.LongTensor(a),
                torch.FloatTensor(r),
                torch.FloatTensor(np.array(ns)),
                torch.FloatTensor(d))

    def __len__(self):
        return len(self.buf)

# ── 3. DQN Agent ───────────────────────────────────────────
class DQNAgent:
    def __init__(self, state_dim, action_dim,
                 lr=5e-4, gamma=0.99, batch_size=128,
                 buffer_size=20_000, target_update_freq=200):
        self.action_dim  = action_dim
        self.gamma       = gamma
        self.batch_size  = batch_size
        self.update_freq = target_update_freq
        self.step_count  = 0

        # Online network (learns every step)
        self.q_net    = QNetwork(state_dim, action_dim)
        # Target network (updated every C steps — provides stable targets)
        self.q_target = QNetwork(state_dim, action_dim)
        self.q_target.load_state_dict(self.q_net.state_dict())
        self.q_target.eval()   # Target never receives gradients

        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.buffer    = ReplayBuffer(buffer_size)

    def select_action(self, state, epsilon):
        """ε-greedy action selection."""
        if random.random() < epsilon:
            return random.randrange(self.action_dim)
        with torch.no_grad():
            q = self.q_net(torch.FloatTensor(state))
        return int(q.argmax())

    def update(self):
        """One gradient step on a random mini-batch."""
        if len(self.buffer) < self.batch_size:
            return None

        s, a, r, ns, done = self.buffer.sample(self.batch_size)

        # Current Q-values: Q(s, a) for the actions actually taken
        current_q = self.q_net(s).gather(1, a.unsqueeze(1)).squeeze()

        # TD Target: r + γ * max_a' Q_target(s', a')  (detached — no gradient through target)
        with torch.no_grad():
            next_q    = self.q_target(ns).max(1)[0]
            td_target = r + self.gamma * next_q * (1 - done)

        # ─── Huber loss (more robust than MSE for RL) ───────────────
        loss = F.smooth_l1_loss(current_q, td_target)

        self.optimizer.zero_grad()
        loss.backward()
        # Gradient clipping: prevents exploding gradients (common in deep RL)
        nn.utils.clip_grad_norm_(self.q_net.parameters(), 10.0)
        self.optimizer.step()

        # ─── Sync target network every C steps ──────────────────────
        self.step_count += 1
        if self.step_count % self.update_freq == 0:
            self.q_target.load_state_dict(self.q_net.state_dict())

        return loss.item()

# ── 4. Training Loop ───────────────────────────────────────
def train_dqn(n_episodes=1500, epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995):
    env   = gym.make('CartPole-v1')
    agent = DQNAgent(state_dim=4, action_dim=2)

    epsilon       = epsilon_start
    all_rewards   = []
    solved_at     = None

    print("=" * 60)
    print("DQN on CartPole-v1")
    print("=" * 60)
    print(f"  Goal: average reward ≥ 475 over 50 consecutive episodes")
    print(f"\n  {'Episode':>8}  {'Reward':>8}  {'Avg(50)':>9}  {'ε':>7}")
    print("  " + "-" * 38)

    for ep in range(1, n_episodes + 1):
        state, _ = env.reset(seed=ep)
        total_r  = 0
        done     = False

        while not done:
            action              = agent.select_action(state, epsilon)
            next_state, r, te, tr, _ = env.step(action)
            done                = te or tr
            agent.buffer.push(state, action, r, next_state, float(done))
            agent.update()
            state     = next_state
            total_r  += r

        epsilon     = max(epsilon_end, epsilon * epsilon_decay)
        all_rewards.append(total_r)
        avg50 = np.mean(all_rewards[-50:])

        if ep % 150 == 0:
            print(f"  {ep:>8}  {total_r:>8.0f}  {avg50:>9.1f}  {epsilon:>7.3f}")

        if avg50 >= 475 and solved_at is None:
            solved_at = ep
            print(f"\n  *** SOLVED at episode {ep}! (avg reward = {avg50:.1f}) ***\n")

    env.close()
    if solved_at is None:
        best = max(np.convolve(all_rewards, np.ones(50)/50, mode='valid'))
        print(f"\n  Training complete ({n_episodes} episodes). Best avg50 = {best:.1f}")
    return agent, all_rewards

dqn_agent, dqn_rewards = train_dqn(n_episodes=1500)


DQN on CartPole-v1
  Goal: average reward ≥ 475 over 50 consecutive episodes

   Episode    Reward    Avg(50)        ε
  --------------------------------------
       150        66       44.9    0.471
       300        71      126.5    0.222
       450         9       36.0    0.105
       600        81      100.7    0.049
       750        20      149.7    0.023
       900       500      265.0    0.011
      1050       363      351.5    0.010
      1200        12      280.2    0.010
      1350       118      429.2    0.010
      1500        11      234.8    0.010

  Training complete (1500 episodes). Best avg50 = 460.3


### 2.1 DQN — What We Just Built

Let's break down the key design decisions:

| Component | Without it | With it |
|---|---|---|
| **Experience Replay** | Train on correlated sequential data → diverges | Random mini-batches → stable gradients |
| **Target Network** | Target moves every step → chasing a moving target | Target frozen C steps → stable learning signal |
| **Gradient Clipping** | Catastrophic gradient explosions possible | Network weights stay bounded |
| **Huber Loss** | Outlier transitions dominate MSE loss | Robust to occasional large TD errors |

### Common DQN Extensions

| Extension | What it fixes | How |
|---|---|---|
| **Double DQN** | Overestimation bias in max operator | Separate networks for action *selection* vs *evaluation* |
| **Dueling DQN** | Inefficient learning when action doesn't matter | Splits Q into V(s) + A(s,a); learns state value faster |
| **Prioritized Replay** | Rare important transitions sampled too rarely | Sample proportional to TD error magnitude |
| **Noisy Networks** | ε-greedy is a crude exploration method | Replace ε-greedy with learned stochastic layers |

DQN and its extensions are powerful for **discrete action spaces** (Atari, etc.).  
But they struggle with **continuous actions** (robotics, physics simulations).  
That's where **Policy Gradient** methods shine!


## 3. Policy Gradient Methods — Learn the Policy Directly

DQN learns a **Q-function** and derives a policy by taking argmax. What if we skip the middleman and learn a **policy directly**?

### The Key Insight

Instead of learning Q(s, a) and deriving π, let's parameterize the policy itself:

$$\pi_\theta(a|s) = P(\text{take action } a \text{ in state } s \text{ | parameters } \theta)$$

For discrete actions, this is a **softmax over logits** — a probability distribution over all actions.

### Why is this better than DQN?

| | DQN (Value-Based) | Policy Gradient |
|---|---|---|
| **Output** | Q-values → greedy action | Direct action probabilities |
| **Action space** | Discrete only | Discrete **and** continuous |
| **Stochastic policies** | Requires ε-greedy hack | Naturally stochastic |
| **Convergence** | Can oscillate (chasing moving target) | Guaranteed to converge to local optimum |

### The Policy Gradient Theorem

The gradient of expected return with respect to policy parameters θ is:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}\left[ G_t \cdot \nabla_\theta \log \pi_\theta(a_t | s_t) \right]$$

In plain English: *"Move θ in the direction that makes good actions (high G) more probable."*

- If action $a_t$ led to high return $G_t$: **increase** $\log \pi_\theta(a_t|s_t)$ → action becomes more likely
- If action $a_t$ led to low/negative return $G_t$: **decrease** $\log \pi_\theta(a_t|s_t)$ → action becomes less likely

### 3.1 REINFORCE — The Simplest Policy Gradient

REINFORCE (Williams, 1992) is the most straightforward implementation of the policy gradient theorem:

```
REINFORCE Algorithm:
  Repeat:
    1. Run one full episode using current policy π_θ
    2. Compute returns G_t = Σ γ^k r_{t+k} for each step t
    3. Gradient update: θ ← θ + α · G_t · ∇_θ log π_θ(a_t | s_t)
    4. Discard episode (on-policy: must use fresh data each time)
```

Notice the similarity to Monte Carlo from Part 3 — REINFORCE also waits for a complete episode!


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import numpy as np
import gymnasium as gym

# ============================================================
# REINFORCE on CartPole-v1
# ============================================================

class PolicyNetwork(nn.Module):
    """
    Outputs a PROBABILITY DISTRIBUTION over actions (not Q-values).
    Uses softmax to turn logits into valid probabilities.
    """
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, action_dim)
        )

    def forward(self, x):
        logits = self.net(x)
        return Categorical(logits=logits)   # Returns a distribution object

def compute_returns(rewards, gamma=0.99):
    """Compute discounted returns G_t = r_t + γr_{t+1} + γ²r_{t+2} + ..."""
    G, returns = 0, []
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    returns = torch.FloatTensor(returns)
    # ── Normalize returns to reduce variance ───────────────
    # (baseline trick: subtracting mean doesn't change the gradient direction
    #  but dramatically reduces variance — key to REINFORCE stability)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    return returns

def train_reinforce(n_episodes=1000, lr=1e-3, gamma=0.99):
    env    = gym.make('CartPole-v1')
    policy = PolicyNetwork(state_dim=4, action_dim=2)
    opt    = optim.Adam(policy.parameters(), lr=lr)

    all_rewards = []
    solved_at   = None

    print("=" * 60)
    print("REINFORCE (Policy Gradient) on CartPole-v1")
    print("=" * 60)
    print(f"  Goal: average reward ≥ 475 over 50 consecutive episodes")
    print(f"\n  {'Episode':>8}  {'Reward':>8}  {'Avg(50)':>9}  {'Loss':>10}")
    print("  " + "-" * 42)

    for ep in range(1, n_episodes + 1):
        state, _ = env.reset(seed=ep)
        log_probs, rewards = [], []
        done = False

        # ── Step 1: Generate one complete episode ────────────
        while not done:
            state_t  = torch.FloatTensor(state)
            dist     = policy(state_t)          # Get action distribution
            action   = dist.sample()             # Sample action from π_θ(·|s)
            log_prob = dist.log_prob(action)     # log π_θ(a|s) for gradient

            state, reward, te, tr, _ = env.step(action.item())
            done = te or tr
            log_probs.append(log_prob)
            rewards.append(reward)

        # ── Step 2: Compute returns G_t ──────────────────────
        returns = compute_returns(rewards, gamma)

        # ── Step 3: Policy gradient update ───────────────────
        # Loss = -E[G_t · log π_θ(a_t|s_t)]
        # Negative because we MAXIMIZE expected return → PyTorch MINIMIZES loss
        loss = -torch.stack(log_probs)
        loss = (loss * returns).mean()

        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        opt.step()

        all_rewards.append(sum(rewards))
        avg50 = np.mean(all_rewards[-50:])

        if ep % 100 == 0:
            print(f"  {ep:>8}  {sum(rewards):>8.0f}  {avg50:>9.1f}  {loss.item():>10.4f}")

        if avg50 >= 475 and solved_at is None:
            solved_at = ep
            print(f"\n  *** SOLVED at episode {ep}! (avg reward = {avg50:.1f}) ***\n")

    env.close()
    if solved_at is None:
        print(f"\n  Training complete. Best avg50 = {max(np.convolve(all_rewards, np.ones(50)/50, mode='valid')):.1f}")
    return policy, all_rewards

reinforce_policy, reinforce_rewards = train_reinforce(n_episodes=1000)


REINFORCE (Policy Gradient) on CartPole-v1
  Goal: average reward ≥ 475 over 50 consecutive episodes

   Episode    Reward    Avg(50)        Loss
  ------------------------------------------
       100        99       56.0     -0.0541
       200       150      232.0      0.0225
       300       118      269.7     -0.0094
       400       490      433.0     -0.0071

  *** SOLVED at episode 488! (avg reward = 476.7) ***

       500       320      490.3     -0.0124
       600       500      480.7      0.0167
       700       500      490.6     -0.0162
       800       500      438.9     -0.0179
       900       118      473.3     -0.0035
      1000       500      499.6      0.0123


## 4. Actor-Critic — The Bridge to Modern Deep RL

REINFORCE has a deep flaw: **high variance**. The return $G_t$ fluctuates wildly across episodes. Imagine teaching a student by only telling them "you did well this episode" or "you did poorly" — no specific feedback about *which* decisions were good.

**Actor-Critic** fixes this by learning both a policy *and* a value function simultaneously:

```
    ACTOR  π_θ(a|s)       — The Policy: decides which action to take
       ↑                                          ↑
   "Which action            "The Advantage A(s,a) tells
   should I take?"    →      you how much better or worse
                             this action was vs average"
       ↑
    CRITIC  V_φ(s)         — The Value Function: evaluates how good the state is
```

### The Advantage Function

Instead of using raw return $G_t$ in the gradient, Actor-Critic uses the **Advantage**:

$$A(s_t, a_t) = Q(s_t, a_t) - V(s_t)$$

In practice, we estimate it as:
$$A(s_t, a_t) \approx r_t + \gamma V_\phi(s_{t+1}) - V_\phi(s_t) \quad \text{(TD Residual)}$$

> *"Was this action better than what I'd normally expect in this state?"*

- $A > 0$: this action was **better** than average → increase its probability
- $A < 0$: this action was **worse** than average → decrease its probability

The gradient update becomes:
$$\nabla_\theta J \approx A(s_t, a_t) \cdot \nabla_\theta \log \pi_\theta(a_t | s_t)$$

This has **much lower variance** than using raw $G_t$!

### Actor-Critic Family

```
        Actor-Critic
             |
    ┌────────┼────────┐
    |        |        |
   A2C      A3C      GAE
(Advantage  (Async  (Generalized
  A-C)      A-C)   Advantage Est.)
             |
            PPO ← This is what we build next!
```


## 5. PPO — Proximal Policy Optimization

PPO (Schulman et al., 2017 — OpenAI) is today's **most widely used deep RL algorithm**. It powers:
- OpenAI Five (Dota 2)
- ChatGPT's RLHF tuning stage
- OpenAI's robotics manipulation research
- Many production RL systems

### The Problem PPO Solves

REINFORCE and basic Actor-Critic suffer from **policy update instability**:

> *"If you take too large a gradient step, the new policy π_new can be so different from π_old that the old experience data is now useless — or worse, the policy collapses."*

Standard gradient ascent has no built-in mechanism to prevent a huge policy update.

### PPO's Solution: Clip the Update

PPO introduces a **clipped surrogate objective** that prevents large policy changes:

$$\mathcal{L}^{\text{CLIP}}(\theta) = \mathbb{E}_t \left[ \min\left( r_t(\theta) \cdot A_t, \quad \text{clip}(r_t(\theta), 1-\varepsilon, 1+\varepsilon) \cdot A_t \right) \right]$$

where the **probability ratio** is:

$$r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_\text{old}}(a_t|s_t)}$$

**Intuition**:
```
If advantage A_t > 0  (action was GOOD):
  r_t = π_new/π_old > 1  means we're increasing probability of this action
  CLIP at (1+ε):  don't increase probability too aggressively!

If advantage A_t < 0  (action was BAD):
  r_t = π_new/π_old < 1  means we're decreasing probability
  CLIP at (1-ε):  don't decrease probability too aggressively!

ε = 0.2 is the standard value  (allow at most 20% policy change per step)
```

### PPO Full Objective

$$\mathcal{L}(\theta) = \mathcal{L}^{\text{CLIP}} - c_1 \mathcal{L}^{\text{VF}} + c_2 S[\pi_\theta]$$

| Term | Role |
|---|---|
| $\mathcal{L}^{\text{CLIP}}$ | Policy loss (clipped surrogate) |
| $c_1 \mathcal{L}^{\text{VF}}$ | Value function loss (trains the critic) |
| $c_2 S[\pi_\theta]$ | Entropy bonus (encourages exploration) |

### PPO vs TRPO

**TRPO** (Trust Region Policy Optimization, Schulman et al., 2015) solves the same problem but uses a **hard KL divergence constraint**:

$$\max_\theta J(\theta) \quad \text{subject to} \quad D_\text{KL}(\pi_\text{old} \| \pi_\theta) \leq \delta$$

This requires second-order optimization (conjugate gradient + Hessian), making it **computationally expensive** and hard to implement.

**PPO** approximates TRPO's guarantee with the simple clip trick — **first-order optimization** only. Nearly the same performance at a fraction of the complexity. That's why PPO replaced TRPO in practice.

```
TRPO:  Hard constraint via second-order optimization
       → Theoretically rigorous   → Computationally expensive

PPO:   Soft constraint via clipping heuristic
       → Slightly less rigorous   → Fast, simple, widely used
```


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import numpy as np
import gymnasium as gym

# ============================================================
# PPO (Proximal Policy Optimization) on CartPole-v1
# ============================================================

class ActorCritic(nn.Module):
    """
    Shared-backbone Actor-Critic network.
    One network → two heads: policy (actor) + value (critic).
    Sharing layers is efficient and helps the two tasks learn together.
    """
    def __init__(self, state_dim, action_dim, hidden=64):
        super().__init__()
        # Shared feature extractor
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh()
        )
        self.actor  = nn.Linear(hidden, action_dim)  # Policy head
        self.critic = nn.Linear(hidden, 1)            # Value head

    def forward(self, x):
        features    = self.shared(x)
        logits      = self.actor(features)
        value       = self.critic(features).squeeze(-1)
        return Categorical(logits=logits), value

def compute_gae(rewards, values, dones, next_value, gamma=0.99, lam=0.95):
    """
    Generalized Advantage Estimation (GAE) — Schulman 2015.
    Trades off bias vs variance with λ parameter:
      λ=0: pure 1-step TD (low variance, high bias)
      λ=1: pure MC returns (high variance, low bias)
      λ=0.95: sweet spot used by most PPO implementations
    """
    advantages = []
    gae = 0
    for t in reversed(range(len(rewards))):
        if t == len(rewards) - 1:
            next_v = next_value
        else:
            next_v = values[t + 1]
        delta = rewards[t] + gamma * next_v * (1 - dones[t]) - values[t]
        gae   = delta + gamma * lam * (1 - dones[t]) * gae
        advantages.insert(0, gae)
    advantages = torch.FloatTensor(advantages)
    returns    = advantages + torch.FloatTensor(values)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    return advantages, returns

def train_ppo(n_iterations=500, n_steps=1024, n_epochs=10,
              batch_size=64, lr=3e-4, gamma=0.99, lam=0.95,
              clip_eps=0.2, c1=0.5, c2=0.01):
    """
    PPO Training Loop.

    n_steps    : collect this many env steps before each update
    n_epochs   : how many gradient passes over the collected data
    clip_eps   : ε in the clipped surrogate objective (default 0.2)
    c1         : value loss coefficient
    c2         : entropy bonus coefficient (encourages exploration)
    """
    env    = gym.make('CartPole-v1')
    model  = ActorCritic(state_dim=4, action_dim=2)
    opt    = optim.Adam(model.parameters(), lr=lr)

    print("=" * 65)
    print("PPO (Proximal Policy Optimization) on CartPole-v1")
    print("=" * 65)
    print(f"  Hyperparams: clip_ε={clip_eps}, γ={gamma}, λ={lam}")
    print(f"               rollout={n_steps} steps, {n_epochs} epochs, batch={batch_size}")
    print(f"\n  {'Iter':>6}  {'Avg Reward':>12}  {'Policy Loss':>13}  {'Value Loss':>12}  {'Entropy':>9}")
    print("  " + "-" * 58)

    all_rewards = []
    solved_at   = None

    for iteration in range(1, n_iterations + 1):

        # ── Phase 1: Collect Rollout ──────────────────────────
        states, actions, rewards, dones, old_log_probs, values_list = [], [], [], [], [], []
        state, _ = env.reset(seed=iteration)
        ep_reward = 0
        iter_rewards = []

        for step in range(n_steps):
            state_t = torch.FloatTensor(state)
            with torch.no_grad():
                dist, value = model(state_t)
            action   = dist.sample()
            log_prob = dist.log_prob(action)

            next_s, reward, te, tr, _ = env.step(action.item())
            done = te or tr

            states.append(state)
            actions.append(action.item())
            rewards.append(reward)
            dones.append(float(done))
            old_log_probs.append(log_prob.item())
            values_list.append(value.item())

            ep_reward += reward
            if done:
                iter_rewards.append(ep_reward)
                ep_reward = 0
                state, _ = env.reset()
            else:
                state = next_s

        # Bootstrap value for the last incomplete episode
        with torch.no_grad():
            _, next_val = model(torch.FloatTensor(state))

        advantages, returns = compute_gae(rewards, values_list, dones, next_val.item(), gamma, lam)

        states_t    = torch.FloatTensor(np.array(states))
        actions_t   = torch.LongTensor(actions)
        old_lp_t    = torch.FloatTensor(old_log_probs)

        # ── Phase 2: PPO Update (multiple epochs over same data) ─
        p_losses, v_losses, entropies = [], [], []
        for epoch in range(n_epochs):
            indices = torch.randperm(n_steps)
            for start in range(0, n_steps, batch_size):
                idx = indices[start:start + batch_size]

                dist, values = model(states_t[idx])
                new_log_probs = dist.log_prob(actions_t[idx])
                entropy       = dist.entropy().mean()

                # ─── Clipped Surrogate Loss ─────────────────────────
                ratio    = torch.exp(new_log_probs - old_lp_t[idx])
                surr1    = ratio * advantages[idx]
                surr2    = torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * advantages[idx]
                pol_loss = -torch.min(surr1, surr2).mean()

                # ─── Value Loss ──────────────────────────────────────
                val_loss = nn.functional.mse_loss(values, returns[idx])

                # ─── Total Loss ──────────────────────────────────────
                loss = pol_loss + c1 * val_loss - c2 * entropy

                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                opt.step()

                p_losses.append(pol_loss.item())
                v_losses.append(val_loss.item())
                entropies.append(entropy.item())

        avg_r = np.mean(iter_rewards) if iter_rewards else 0
        all_rewards.extend(iter_rewards)
        avg50 = np.mean(all_rewards[-50:]) if len(all_rewards) >= 50 else np.mean(all_rewards)

        if iteration % 50 == 0 or iteration <= 5:
            print(f"  {iteration:>6}  {avg_r:>12.1f}  {np.mean(p_losses):>13.5f}  "
                  f"{np.mean(v_losses):>12.3f}  {np.mean(entropies):>9.4f}")

        if avg50 >= 475 and solved_at is None and len(all_rewards) >= 50:
            solved_at = iteration
            print(f"\n  *** SOLVED at iteration {iteration}! (avg50 = {avg50:.1f}) ***\n")

    env.close()
    final_avg = np.mean(all_rewards[-50:]) if len(all_rewards) >= 50 else np.mean(all_rewards) if all_rewards else 0
    print(f"\n  Final avg reward (last 50 episodes) = {final_avg:.1f}")
    return model, all_rewards

ppo_model, ppo_rewards = train_ppo(n_iterations=500)


PPO (Proximal Policy Optimization) on CartPole-v1
  Hyperparams: clip_ε=0.2, γ=0.99, λ=0.95
               rollout=1024 steps, 10 epochs, batch=64

    Iter    Avg Reward    Policy Loss    Value Loss    Entropy
  ----------------------------------------------------------
       1          19.9       -0.00377        54.684     0.6899
       2          20.0       -0.00486        34.208     0.6845
       3          19.5       -0.00763        31.341     0.6864
       4          24.3       -0.01165        34.853     0.6772
       5          27.8       -0.00369        47.158     0.6663
      50         129.0       -0.00790        68.970     0.5509
     100         143.7        0.00247        21.914     0.5884
     150         202.7        0.00866        63.039     0.5842
     200         292.0        0.00817        52.531     0.5427
     250          94.2       -0.00051       162.653     0.4885
     300         494.0        0.00868        63.111     0.4530
     350         348.0        0.013

## 6. TRPO — Trust Region Policy Optimization

TRPO (Schulman et al., 2015) was the **theoretical predecessor to PPO**. Understanding it helps you understand *why* PPO was designed the way it was.

### The Core Problem TRPO Solves

Every policy gradient update can be framed as:

$$\max_\theta \; \mathcal{L}(\theta) \quad \text{(maximize expected advantage)}$$

But taking large steps in parameter space can **destroy** a good policy. The neural network's parameters can jump to a region where the policy is completely different.

TRPO formalizes this as a **constrained optimization problem**:

$$\max_\theta \mathbb{E}_t\!\left[\frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_\text{old}}(a_t|s_t)} A_t\right] \quad \text{subject to} \quad D_{\text{KL}}(\pi_{\theta_\text{old}} \| \pi_\theta) \leq \delta$$

The constraint says: *"Don't let the new policy differ from the old one by more than δ in KL divergence."*

### TRPO vs PPO at a Glance

```
TRPO (2015)                              PPO (2017)
────────────────────────────             ────────────────────────────
Hard KL constraint:                      Soft clipping:
  D_KL(π_old ‖ π_new) ≤ δ                  clip(r_t, 1-ε, 1+ε)

Requires:                                Requires:
  Conjugate Gradient method               Standard SGD / Adam
  Fisher Information Matrix inverse       No second-order info needed
  Line search to satisfy constraint       Just gradient clipping

Theory:    Strong monotonic improvement   Slightly weaker guarantee
Practice:  Complex  |  Slow               Simple  |  Fast  |  Robust
```

### Why TRPO Still Matters

TRPO proved that you can guarantee **monotonic policy improvement** under certain conditions. This theoretical result underpins the entire trust-region approach to RL and influenced TRPO's numerous successors. Even if you never implement TRPO yourself, understanding its **intuition** — *"don't change the policy too much in one step"* — is crucial for understanding PPO, SAC, and virtually all modern deep RL algorithms.

### Key Algorithms You Should Know (Beyond This Course)

| Algorithm | Year | Key Innovation | Best For |
|---|---|---|---|
| **A3C** | 2016 | Asynchronous parallel workers | Multi-CPU environments |
| **DDPG** | 2016 | Continuous actions with DQN-style replay | Robotics |
| **TRPO** | 2015 | Monotonic improvement guarantee | Theory-critical applications |
| **PPO** | 2017 | Clipped objective ≈ TRPO but simple | General purpose (SOTA default) |
| **SAC** | 2018 | Maximum entropy + off-policy PG | Sample efficiency + stability |
| **TD3** | 2018 | Double critics + delayed policy updates | Continuous control |


## 7. Comparing All Three Deep RL Algorithms

Let's put DQN, REINFORCE, and PPO head-to-head on CartPole and examine their learning curves.


In [8]:
import numpy as np

# ============================================================
# Side-by-Side Comparison: DQN vs REINFORCE vs PPO
# ============================================================

def smooth(data, window=50):
    """Rolling average with a given window size."""
    if len(data) < window:
        return data
    return np.convolve(data, np.ones(window) / window, mode='valid')

def summarize(name, rewards, window=50):
    """Print a performance table for one algorithm."""
    smoothed = smooth(rewards, window)
    chunks   = [rewards[i:i+100] for i in range(0, len(rewards), 100) if len(rewards[i:i+100]) >= 10]
    print(f"\n  {name}")
    print(f"  {'─'*55}")
    print(f"  {'Episode Range':>16}  {'Avg Reward':>12}  {'Min':>8}  {'Max':>8}")
    print(f"  {'─'*55}")
    for i, chunk in enumerate(chunks[:8]):  # show first 800 episodes
        start = i * 100 + 1
        end   = start + len(chunk) - 1
        print(f"  {start:>7} - {end:<7}   {np.mean(chunk):>10.1f}  {min(chunk):>8.0f}  {max(chunk):>8.0f}")
    if len(smoothed) > 0:
        ep_solved = None
        for i, v in enumerate(smoothed):
            if v >= 475:
                ep_solved = i + window
                break
        best = max(smoothed) if len(smoothed) > 0 else 0
        print(f"\n  Best smoothed avg : {best:.1f}")
        if ep_solved:
            print(f"  First solved at ep: {ep_solved}")
        else:
            print(f"  Not solved in {len(rewards)} episodes")

print("=" * 65)
print("ALGORITHM COMPARISON: DQN vs REINFORCE vs PPO on CartPole-v1")
print("=" * 65)
print("(Rewards taken from training runs above)")

summarize("DQN   (Value-Based, Off-Policy)",   dqn_rewards)
summarize("REINFORCE (Policy Gradient, On-Policy)", reinforce_rewards)
summarize("PPO   (Actor-Critic, On-Policy)",   ppo_rewards)

print()
print("=" * 65)
print("KEY TAKEAWAYS")
print("=" * 65)
print("""
  DQN:       Sample efficient (reuses experience), fast on GPU,
             but limited to DISCRETE actions.
             Hard to tune: batch size, buffer size, target update freq.

  REINFORCE: Simple, theoretically clean, handles any action space.
             HIGH VARIANCE → needs many episodes to converge.
             Not suitable for complex tasks without variance reduction.

  PPO:       Best of both worlds — stable like DQN, flexible like PG.
             Clips updates → safe, reproducible training.
             Industry standard for most modern RL problems.
             (Used for RLHF in GPT, robotics, games, etc.)
""")

print("Algorithm Decision Guide:")
print("-" * 65)
print(f"  {'Situation':<40} {'Use'}")
print("-" * 65)
guide = [
    ("Discrete actions, sample efficiency matters", "DQN / Double DQN"),
    ("Continuous actions (robotics, physics)",       "PPO / SAC / TD3"),
    ("Simple env, teaching/demo purposes",           "REINFORCE"),
    ("Complex env, need reliability + speed",        "PPO"),
    ("Off-policy, replay buffer, continuous",        "SAC"),
    ("Need theoretical guarantees",                  "TRPO"),
    ("Huge state space (Atari pixels)",              "DQN / Rainbow"),
    ("RLHF tuning a language model",                 "PPO"),
]
for sit, alg in guide:
    print(f"  {sit:<40} {alg}")


ALGORITHM COMPARISON: DQN vs REINFORCE vs PPO on CartPole-v1
(Rewards taken from training runs above)

  DQN   (Value-Based, Off-Policy)
  ───────────────────────────────────────────────────────
     Episode Range    Avg Reward       Min       Max
  ───────────────────────────────────────────────────────
        1 - 100             33.5         8       144
      101 - 200             48.0        10       279
      201 - 300            107.7         9       364
      301 - 400             67.7        10       414
      401 - 500             40.6         9        99
      501 - 600             89.3        18       151
      601 - 700             80.9        11       104
      701 - 800            119.9        10       500

  Best smoothed avg : 460.3
  Not solved in 1500 episodes

  REINFORCE (Policy Gradient, On-Policy)
  ───────────────────────────────────────────────────────
     Episode Range    Avg Reward       Min       Max
  ───────────────────────────────────────────────────────


## 8. ✏️ Exercise: Tune PPO on LunarLander-v3

LunarLander is a significantly harder environment than CartPole:

```
┌─────────────────────────────────────────────────────┐
│  LunarLander-v3                                     │
│                                                     │
│  Goal: Land a rocket between two flags              │
│                                                     │
│  State (8 values):                                  │
│    [x, y, vx, vy, angle, angular_vel, leg1, leg2]   │
│                                                     │
│  Actions (discrete):                                │
│    0 = Do nothing                                   │
│    1 = Fire left engine                             │
│    2 = Fire main engine                             │
│    3 = Fire right engine                            │
│                                                     │
│  Reward:                                            │
│    +100 to +140 for successful landing              │
│    -100 for crashing                                │
│    -0.3 for each main engine firing                 │
│    +10 for each leg touching ground                 │
│                                                     │
│  Solved: avg reward ≥ 200 over 100 episodes         │
└─────────────────────────────────────────────────────┘
```

### Your Tasks

1. Run the PPO implementation below on LunarLander (change `env_name` and `state_dim`/`action_dim`)
2. Try at least **two different values** of `clip_eps` (e.g., 0.1 and 0.3) — which works better?
3. Tune `n_steps` (try 256 vs 2048) — what effect does rollout length have?
4. Try removing the entropy bonus (`c2=0`) — what happens to exploration?

The cell below has `TODO` markers for your modifications:


In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import numpy as np
import gymnasium as gym

# ============================================================
# Exercise: PPO on LunarLander-v3
# ============================================================
# Modify the TODO parameters below and observe the effect.

# TODO 1 — Change these hyperparameters and compare results:
CLIP_EPS  = 0.2    # TODO: try 0.1 or 0.3
N_STEPS   = 1024   # TODO: try 256 or 2048 (rollout length)
ENTROPY_C = 0.01   # TODO: try 0.0 (no entropy) or 0.05

# TODO 2 — You can also try tweaking these:
LR        = 3e-4
N_EPOCHS  = 4
BATCH_SZ  = 64
GAMMA     = 0.99
LAM       = 0.95

# ── Reuse ActorCritic and compute_gae from above ──────────

def train_ppo_lunarlander(n_iterations=300):
    env    = gym.make('LunarLander-v3')
    state_dim  = env.observation_space.shape[0]   # 8
    action_dim = env.action_space.n               # 4
    model  = ActorCritic(state_dim, action_dim, hidden=128)
    opt    = optim.Adam(model.parameters(), lr=LR)

    print("=" * 65)
    print(f"PPO on LunarLander-v3  (clip_ε={CLIP_EPS}, steps={N_STEPS}, H={ENTROPY_C})")
    print("=" * 65)
    print(f"  State dim={state_dim}, Action dim={action_dim}")
    print(f"  Goal: avg reward ≥ 200 over 100 episodes\n")
    print(f"  {'Iter':>6}  {'Avg Reward':>12}  {'Policy Loss':>13}  {'Value Loss':>12}")
    print("  " + "-" * 50)

    all_rewards = []
    solved_at   = None

    for iteration in range(1, n_iterations + 1):
        states, actions, rewards, dones, old_log_probs, values_list = [], [], [], [], [], []
        state, _ = env.reset(seed=iteration)
        ep_reward = 0
        iter_rewards = []

        for step in range(N_STEPS):
            state_t = torch.FloatTensor(state)
            with torch.no_grad():
                dist, value = model(state_t)
            action   = dist.sample()
            log_prob = dist.log_prob(action)
            next_s, reward, te, tr, _ = env.step(action.item())
            done = te or tr

            states.append(state)
            actions.append(action.item())
            rewards.append(reward)
            dones.append(float(done))
            old_log_probs.append(log_prob.item())
            values_list.append(value.item())

            ep_reward += reward
            if done:
                iter_rewards.append(ep_reward)
                ep_reward = 0
                state, _ = env.reset()
            else:
                state = next_s

        with torch.no_grad():
            _, next_val = model(torch.FloatTensor(state))

        advantages, returns = compute_gae(rewards, values_list, dones, next_val.item(), GAMMA, LAM)
        states_t  = torch.FloatTensor(np.array(states))
        actions_t = torch.LongTensor(actions)
        old_lp_t  = torch.FloatTensor(old_log_probs)

        p_losses, v_losses = [], []
        for _ in range(N_EPOCHS):
            indices = torch.randperm(N_STEPS)
            for start in range(0, N_STEPS, BATCH_SZ):
                idx = indices[start:start + BATCH_SZ]
                dist, values = model(states_t[idx])
                new_lp  = dist.log_prob(actions_t[idx])
                entropy = dist.entropy().mean()
                ratio   = torch.exp(new_lp - old_lp_t[idx])
                surr1   = ratio * advantages[idx]
                surr2   = torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * advantages[idx]
                pol_loss = -torch.min(surr1, surr2).mean()
                val_loss = nn.functional.mse_loss(values, returns[idx])
                loss = pol_loss + 0.5 * val_loss - ENTROPY_C * entropy
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                opt.step()
                p_losses.append(pol_loss.item())
                v_losses.append(val_loss.item())

        avg_r = np.mean(iter_rewards) if iter_rewards else 0
        all_rewards.extend(iter_rewards)
        avg100 = np.mean(all_rewards[-100:]) if len(all_rewards) >= 100 else np.mean(all_rewards) if all_rewards else 0

        if iteration % 30 == 0 or iteration <= 5:
            print(f"  {iteration:>6}  {avg_r:>12.1f}  {np.mean(p_losses):>13.5f}  {np.mean(v_losses):>12.3f}")

        if avg100 >= 200 and solved_at is None and len(all_rewards) >= 100:
            solved_at = iteration
            print(f"\n  *** SOLVED at iteration {iteration}! (avg100 = {avg100:.1f}) ***\n")

    env.close()
    final_avg = np.mean(all_rewards[-100:]) if len(all_rewards) >= 100 else np.mean(all_rewards) if all_rewards else 0
    print(f"\n  Training complete. Final avg(last 100) = {final_avg:.1f}")
    if solved_at:
        print(f"  Solved at iteration: {solved_at}")
    else:
        print(f"  Not solved in {n_iterations} iterations — try more iterations or tuning!")

    return model, all_rewards

lunar_model, lunar_rewards = train_ppo_lunarlander(n_iterations=300)

print("\n" + "─" * 65)
print("DISCUSSION QUESTIONS:")
print("  1. How does clip_eps=0.1 vs 0.3 affect stability and speed of learning?")
print("  2. What happens with n_steps=256?  Fewer steps = noisier estimates.")
print("  3. What happens when entropy_c=0?  No exploration bonus — does it collapse?")
print("  4. LunarLander is harder than CartPole. How many PPO iterations does it take?")


PPO on LunarLander-v3  (clip_ε=0.2, steps=1024, H=0.01)
  State dim=8, Action dim=4
  Goal: avg reward ≥ 200 over 100 episodes

    Iter    Avg Reward    Policy Loss    Value Loss
  --------------------------------------------------
       1        -226.8       -0.00021      2148.666
       2        -159.4       -0.00551      1396.248
       3        -185.7        0.00094      1142.326
       4        -248.5       -0.00319      2918.909
       5        -244.4       -0.00580      2111.412
      30        -163.7        0.00839       515.950
      60        -305.8        0.00006      1567.776
      90        -176.5       -0.00134       375.649
     120        -190.0        0.00070       455.613
     150        -152.3       -0.00122       600.132
     180        -161.3       -0.00408       137.365
     210         -86.8        0.00384       168.122
     240        -238.4       -0.01113       101.069
     270        -184.7       -0.00160       724.118
     300         -62.2       -0.00422  

## 9. Summary & Key Takeaways

### What We Built Today

✅ **DQN** — Implemented from scratch with Experience Replay + Target Network on CartPole; understood why both innovations are critical for stable deep RL training  

✅ **REINFORCE** — Built the fundamental policy gradient algorithm; used the log-derivative trick and return normalization to stabilize training  

✅ **Actor-Critic** — Understood the Advantage function $A(s,a) = Q(s,a) - V(s)$ as a lower-variance replacement for raw returns  

✅ **PPO** — Implemented the clipped surrogate objective, GAE, and multi-epoch updates; the same core algorithm used for RLHF in LLMs  

✅ **TRPO** — Understood the trust-region concept and KL constraint; know why PPO replaced it in practice  

✅ **When to use what** — DQN for discrete actions, PPO for general purpose, SAC for continuous control, REINFORCE for teaching

---

### The Full RL Algorithm Progression

```
Part 3 (Tabular RL)              Part 4 (Deep RL)
────────────────────             ─────────────────────────────

Dynamic Programming              → DQN
(exact, known model)               (approximate, continuous states)

Monte Carlo                      → REINFORCE
(episode returns, no model)        (policy gradient, on-policy)

Q-Learning (off-policy TD)       → DQN (with replay + target net)
                                   → SAC / TD3 (continuous)

SARSA (on-policy TD)             → Actor-Critic → A2C → PPO
                                   (advantage + clipped updates)
```

---

### Key Terms Cheat Sheet

| Term | Plain English |
|---|---|
| **DQN** | Q-Learning with a neural network approximating Q(s,a) |
| **Experience Replay** | Shuffle past transitions to break correlation + allow reuse |
| **Target Network** | Frozen copy of Q-net to provide stable TD targets |
| **Policy Gradient** | Directly optimize policy parameters θ by gradient ascent on J(θ) |
| **REINFORCE** | On-policy PG: run episode → compute returns → gradient update |
| **Advantage A(s,a)** | How much better this action was vs average in this state |
| **Actor-Critic** | Two networks: Actor (policy) + Critic (value function) |
| **GAE** | Generalized Advantage Estimation: controls bias/variance tradeoff |
| **PPO clip** | Prevents ratio r_t = π_new/π_old from straying beyond [1-ε, 1+ε] |
| **TRPO** | Hard KL constraint on policy updates; PPO's theoretical parent |
| **On-policy** | Must train on data from the CURRENT policy (REINFORCE, PPO) |
| **Off-policy** | Can reuse old data via replay buffer (DQN, SAC) |
| **Entropy bonus** | Reward for maintaining action diversity = encourages exploration |

---

### Coming Up in Part 5: Advanced Topics

In Part 5 we'll explore:
1. **Multi-Agent RL** — multiple agents interacting in the same environment
2. **Model-Based RL** — learn a world model and plan inside it (Dyna-Q, MuZero)
3. **RLHF** — Reinforcement Learning from Human Feedback (the ChatGPT training pipeline in detail)
4. **Reward Shaping & Sparse Reward Problems** — what to do when the agent almost never sees a reward

---

*© 2025 Reinforcement Learning for AI/ML Engineers — Mehdi Lotfinejad*


## 📋 Quick Reference Card

```
┌─────────────────────────────────────────────────────────────────────┐
│              DEEP RL ALGORITHMS CHEAT SHEET                         │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  DQN (Deep Q-Network)                                               │
│  ┌─────────────────────────────────────────────────────────────┐   │
│  │  Q_θ(s,a) ← neural network approximates Q-table             │   │
│  │  Loss = (r + γ·max Q_θ⁻(s',a') − Q_θ(s,a))²                │   │
│  │  Key: Experience Replay + Target Network + gradient clip     │   │
│  └─────────────────────────────────────────────────────────────┘   │
│                                                                     │
│  REINFORCE (Policy Gradient)                                        │
│  ┌─────────────────────────────────────────────────────────────┐   │
│  │  π_θ(a|s) = softmax(f_θ(s))                                 │   │
│  │  θ ← θ + α · G_t · ∇_θ log π_θ(a_t|s_t)                    │   │
│  │  Key: Run full episode, normalize returns, on-policy         │   │
│  └─────────────────────────────────────────────────────────────┘   │
│                                                                     │
│  Actor-Critic                                                       │
│  ┌─────────────────────────────────────────────────────────────┐   │
│  │  Actor:   π_θ(a|s)   — the policy                           │   │
│  │  Critic:  V_φ(s)     — value estimate                       │   │
│  │  Advantage: A = r + γV(s') − V(s)  (TD residual)            │   │
│  └─────────────────────────────────────────────────────────────┘   │
│                                                                     │
│  PPO (Proximal Policy Optimization)                                 │
│  ┌─────────────────────────────────────────────────────────────┐   │
│  │  r_t(θ) = π_θ(a|s) / π_old(a|s)   (probability ratio)      │   │
│  │  L_CLIP = E[min(r_t·A_t, clip(r_t, 1-ε, 1+ε)·A_t)]         │   │
│  │  Total  = L_CLIP − c₁·L_VF + c₂·S[π]                       │   │
│  │  Key: Collect rollout, compute GAE, update k epochs          │   │
│  └─────────────────────────────────────────────────────────────┘   │
│                                                                     │
│  TRPO (Trust Region Policy Optimization)                            │
│  ┌─────────────────────────────────────────────────────────────┐   │
│  │  max E[r_t·A_t]  s.t.  D_KL(π_old ‖ π_new) ≤ δ            │   │
│  │  Requires: Conjugate Gradient + Fisher matrix + line search  │   │
│  │  PPO approximates this with the clip trick (use PPO instead) │   │
│  └─────────────────────────────────────────────────────────────┘   │
│                                                                     │
│  HYPERPARAMETERS (PPO defaults):                                    │
│  lr     = 3e-4  (learning rate)                                    │
│  γ      = 0.99  (discount factor)                                  │
│  λ      = 0.95  (GAE parameter)                                    │
│  ε      = 0.2   (clip range)                                       │
│  c₁     = 0.5   (value loss coefficient)                           │
│  c₂     = 0.01  (entropy coefficient)                              │
│  steps  = 2048  (rollout length per update)                        │
│  epochs = 10    (passes over rollout data)                         │
│                                                                     │
│  DECISION GUIDE:                                                    │
│  Discrete actions, reuse data?   → DQN                             │
│  Continuous actions?             → PPO / SAC / TD3                 │
│  Simple env, learning purposes?  → REINFORCE                       │
│  Production / complex env?       → PPO (default choice)            │
│  RLHF for LLM?                   → PPO                             │
│  Need theoretical guarantees?    → TRPO                            │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```
